# 论文 3：理解 LSTM 网络
## Christopher Olah

### LSTM 实现与门控可视化

LSTM（长短期记忆）网络通过带门控的记忆单元解决梯度消失问题。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## LSTM 单元实现

LSTM 包含三个门：
1. **遗忘门**：决定从细胞状态中遗忘哪些内容
2. **输入门**：决定加入哪些新信息
3. **输出门**：决定根据细胞状态输出哪些内容

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

class LSTMCell:
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # 为提高效率，使用拼接后的权重：[输入; 隐藏状态] -> 各个门
        concat_size = input_size + hidden_size
        
        # 遗忘门
        self.Wf = np.random.randn(hidden_size, concat_size) * 0.01
        self.bf = np.zeros((hidden_size, 1))
        
        # 输入门
        self.Wi = np.random.randn(hidden_size, concat_size) * 0.01
        self.bi = np.zeros((hidden_size, 1))
        
        # 候选细胞状态
        self.Wc = np.random.randn(hidden_size, concat_size) * 0.01
        self.bc = np.zeros((hidden_size, 1))
        
        # 输出门
        self.Wo = np.random.randn(hidden_size, concat_size) * 0.01
        self.bo = np.zeros((hidden_size, 1))
    
    def forward(self, x, h_prev, c_prev):
        """
        LSTM 单元的前向传播
        
        x：输入，形状为 (input_size, 1)
        h_prev：前一个隐藏状态，形状为 (hidden_size, 1)
        c_prev：前一个细胞状态，形状为 (hidden_size, 1)
        
        返回：
        h_next：下一个隐藏状态
        c_next：下一个细胞状态
        cache：反向传播所需的中间值
        """
        # 拼接当前输入和前一个隐藏状态
        concat = np.vstack([x, h_prev])
        
        # 遗忘门：决定从细胞状态中遗忘哪些内容
        f = sigmoid(np.dot(self.Wf, concat) + self.bf)
        
        # 输入门：决定存储哪些新信息
        i = sigmoid(np.dot(self.Wi, concat) + self.bi)
        
        # 候选细胞状态：可能加入细胞状态的新信息
        c_tilde = np.tanh(np.dot(self.Wc, concat) + self.bc)
        
        # 更新细胞状态：遗忘旧信息，并加入新信息
        c_next = f * c_prev + i * c_tilde
        
        # 输出门：决定输出哪些内容
        o = sigmoid(np.dot(self.Wo, concat) + self.bo)
        
        # 隐藏状态：经过筛选的细胞状态
        h_next = o * np.tanh(c_next)
        
        # 缓存反向传播需要的中间值
        cache = (x, h_prev, c_prev, concat, f, i, c_tilde, c_next, o, h_next)
        
        return h_next, c_next, cache

# 测试 LSTM 单元
input_size = 10
hidden_size = 20
lstm_cell = LSTMCell(input_size, hidden_size)

x = np.random.randn(input_size, 1)
h = np.zeros((hidden_size, 1))
c = np.zeros((hidden_size, 1))

h_next, c_next, cache = lstm_cell.forward(x, h, c)
print(f"LSTM Cell initialized: input_size={input_size}, hidden_size={hidden_size}")
print(f"Hidden state shape: {h_next.shape}")
print(f"Cell state shape: {c_next.shape}")

## 用于序列处理的完整 LSTM 网络

In [ ]:
class LSTM:
    def __init__(self, input_size, hidden_size, output_size):
        self.hidden_size = hidden_size
        self.cell = LSTMCell(input_size, hidden_size)
        
        # 输出层
        self.Why = np.random.randn(output_size, hidden_size) * 0.01
        self.by = np.zeros((output_size, 1))
    
    def forward(self, inputs):
        """
        使用 LSTM 处理整个序列
        inputs：输入向量列表
        """
        h = np.zeros((self.hidden_size, 1))
        c = np.zeros((self.hidden_size, 1))
        
        # 保存各个状态，以便后续可视化
        h_states = []
        c_states = []
        gate_values = {'f': [], 'i': [], 'o': []}
        
        for x in inputs:
            h, c, cache = self.cell.forward(x, h, c)
            h_states.append(h.copy())
            c_states.append(c.copy())
            
            # 从缓存中提取各个门的值
            _, _, _, _, f, i, _, _, o, _ = cache
            gate_values['f'].append(f.copy())
            gate_values['i'].append(i.copy())
            gate_values['o'].append(o.copy())
        
        # 计算最终输出
        y = np.dot(self.Why, h) + self.by
        
        return y, h_states, c_states, gate_values

# 创建 LSTM 模型
input_size = 5
hidden_size = 16
output_size = 5
lstm = LSTM(input_size, hidden_size, output_size)
print(f"\nLSTM model created: {input_size} -> {hidden_size} -> {output_size}")

## 在合成序列任务上测试：长期依赖

任务：记住序列开头的一个值，并在序列末尾将其输出。

In [ ]:
def generate_long_term_dependency_data(seq_length=20, num_samples=100):
    """
    生成序列；模型需要一直记住第一个元素，直到序列结束
    """
    X = []
    y = []
    
    for _ in range(num_samples):
        # 创建序列
        sequence = []
        
        # 第一个元素是需要记忆的重要信息（one-hot 编码）
        first_elem = np.random.randint(0, input_size)
        first_vec = np.zeros((input_size, 1))
        first_vec[first_elem] = 1
        sequence.append(first_vec)
        
        # 其余元素均为随机噪声
        for _ in range(seq_length - 1):
            noise = np.random.randn(input_size, 1) * 0.1
            sequence.append(noise)
        
        X.append(sequence)
        
        # 目标：回忆第一个元素
        target = np.zeros((output_size, 1))
        target[first_elem] = 1
        y.append(target)
    
    return X, y

# 生成测试数据
X_test, y_test = generate_long_term_dependency_data(seq_length=15, num_samples=10)

# 测试前向传播
output, h_states, c_states, gate_values = lstm.forward(X_test[0])

print(f"\nTest sequence length: {len(X_test[0])}")
print(f"First element (to remember): {np.argmax(X_test[0][0])}")
print(f"Expected output: {np.argmax(y_test[0])}")
print(f"Model output (untrained): {output.flatten()[:5]}")

## 可视化 LSTM 的门控机制

理解 LSTM 的关键，是观察各个门如何随时间发挥作用。

In [ ]:
# 处理一个序列，并可视化各个门
test_seq = X_test[0]
output, h_states, c_states, gate_values = lstm.forward(test_seq)

# 转换为数组，以便绘图
forget_gates = np.hstack(gate_values['f'])
input_gates = np.hstack(gate_values['i'])
output_gates = np.hstack(gate_values['o'])
cell_states = np.hstack(c_states)
hidden_states = np.hstack(h_states)

fig, axes = plt.subplots(5, 1, figsize=(14, 12))

# 遗忘门
axes[0].imshow(forget_gates, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[0].set_title('Forget Gate (1=keep, 0=forget)')
axes[0].set_ylabel('Hidden Unit')
axes[0].set_xlabel('Time Step')

# 输入门
axes[1].imshow(input_gates, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[1].set_title('Input Gate (1=accept new, 0=ignore new)')
axes[1].set_ylabel('Hidden Unit')
axes[1].set_xlabel('Time Step')

# 输出门
axes[2].imshow(output_gates, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[2].set_title('Output Gate (1=expose, 0=hide)')
axes[2].set_ylabel('Hidden Unit')
axes[2].set_xlabel('Time Step')

# 细胞状态
im3 = axes[3].imshow(cell_states, cmap='RdBu', aspect='auto')
axes[3].set_title('Cell State (Long-term Memory)')
axes[3].set_ylabel('Hidden Unit')
axes[3].set_xlabel('Time Step')
plt.colorbar(im3, ax=axes[3])

# 隐藏状态
im4 = axes[4].imshow(hidden_states, cmap='RdBu', aspect='auto')
axes[4].set_title('Hidden State (Output to Next Layer)')
axes[4].set_ylabel('Hidden Unit')
axes[4].set_xlabel('Time Step')
plt.colorbar(im4, ax=axes[4])

plt.tight_layout()
plt.show()

print("\nGate Interpretation:")
print("- Forget gate controls what information to discard from cell state")
print("- Input gate controls what new information to add to cell state")
print("- Output gate controls what to output from cell state")
print("- Cell state is the long-term memory highway")

## 比较 LSTM 与普通 RNN 处理长序列的能力

In [ ]:
class VanillaRNNCell:
    def __init__(self, input_size, hidden_size):
        concat_size = input_size + hidden_size
        self.Wh = np.random.randn(hidden_size, concat_size) * 0.01
        self.bh = np.zeros((hidden_size, 1))
        self.hidden_size = hidden_size
    
    def forward(self, x, h_prev):
        concat = np.vstack([x, h_prev])
        h_next = np.tanh(np.dot(self.Wh, concat) + self.bh)
        return h_next

# 创建普通 RNN，用于比较
rnn_cell = VanillaRNNCell(input_size, hidden_size)

def process_with_vanilla_rnn(inputs):
    h = np.zeros((hidden_size, 1))
    h_states = []
    
    for x in inputs:
        h = rnn_cell.forward(x, h)
        h_states.append(h.copy())
    
    return h_states

# 使用两种模型处理同一个序列
rnn_h_states = process_with_vanilla_rnn(test_seq)
rnn_hidden = np.hstack(rnn_h_states)

# 比较隐藏状态的演化过程
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

im1 = ax1.imshow(rnn_hidden, cmap='RdBu', aspect='auto')
ax1.set_title('Vanilla RNN Hidden States')
ax1.set_ylabel('Hidden Unit')
ax1.set_xlabel('Time Step')
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(hidden_states, cmap='RdBu', aspect='auto')
ax2.set_title('LSTM Hidden States')
ax2.set_ylabel('Hidden Unit')
ax2.set_xlabel('Time Step')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

print("\nKey Difference:")
print("- LSTM maintains cell state separate from hidden state")
print("- Gates allow selective information flow")
print("- Better gradient flow through time (solves vanishing gradient)")

## 梯度流比较

In [ ]:
# 模拟梯度大小
def simulate_gradient_flow(seq_length=30):
    """
    模拟普通 RNN 和 LSTM 中的梯度如何衰减
    """
    # 普通 RNN：梯度呈指数衰减
    rnn_grads = []
    grad = 1.0
    decay_factor = 0.85  # 普通 RNN 中具有代表性的衰减系数
    
    for t in range(seq_length):
        rnn_grads.append(grad)
        grad *= decay_factor
    
    # LSTM：通过细胞状态的信息高速通道维持梯度
    lstm_grads = []
    grad = 1.0
    forget_gate_avg = 0.95  # 遗忘门值较高时，梯度得到保留
    
    for t in range(seq_length):
        lstm_grads.append(grad)
        grad *= forget_gate_avg  # 遗忘门控制梯度流
    
    return np.array(rnn_grads), np.array(lstm_grads)

rnn_grads, lstm_grads = simulate_gradient_flow()

plt.figure(figsize=(12, 5))
plt.plot(rnn_grads[::-1], label='Vanilla RNN', linewidth=2)
plt.plot(lstm_grads[::-1], label='LSTM', linewidth=2)
plt.xlabel('Timesteps in the Past')
plt.ylabel('Gradient Magnitude')
plt.title('Gradient Flow: LSTM vs Vanilla RNN')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

print(f"\nGradient after 30 steps:")
print(f"Vanilla RNN: {rnn_grads[-1]:.6f} (vanished)")
print(f"LSTM: {lstm_grads[-1]:.6f} (preserved)")
print(f"\nThis is why LSTM can learn long-term dependencies!")

## 核心要点

### LSTM 架构：
1. **细胞状态**：信息跨时间流动的高速通道
2. **遗忘门**：控制从记忆中移除哪些内容
3. **输入门**：控制加入哪些新信息
4. **输出门**：控制从记忆中输出哪些内容

### LSTM 为什么有效：
- **恒定误差环流（Constant Error Carousel）**：细胞状态让梯度持续流动
- **乘法门控**：让网络学习何时记忆、何时遗忘
- **加法更新**：通过加法更新细胞状态（f*c + i*c_tilde）
- **梯度保持**：遗忘门接近 1 时可以保留梯度

### 相比普通 RNN 的优势：
- 解决梯度消失问题
- 能够学习长期依赖关系（超过 100 个时间步）
- 训练过程更加稳定
- 在真实世界的序列任务上表现更好